# Polynomial-cost orbital optimization of cluster number quasisymetries

In [17]:
import numpy as np

# Molecule input parameters
molecule = 'h2o' # 'n2'      # supports: h2, h2o, n2, lih, h4_linear, h4_square
bond_length = .96 # 1.1     # Angstrom
basis = 'sto3g' # '6-31g'

# DMRG parameters
bond_dim = 50 # 500
n_sweeps = 10 # 20

# High-level input parameters
initial_basis = 'MOs' # 'MOs': HF molecular orbitals, #TODO: 'NOs': natural orbitals, 'localized': orbitals localized on nuclei via...
use_custom_cluster_matrix = True # True: set your cluster matrix below, #TODO False: automatic cluster_matrix optimization, set num_clusters below
num_transfers = 2 # maximal number of inter-cluster electron transfers starting from fundamental sector
type_cost_function = 'variance' # 'variance' or #TODO: 'eval_eq'
if use_custom_cluster_matrix:
    cluster_matrix = np.array([ # num_clusters x norb binary matrix with custom clusters. Clusters should not overlap nor use all orbitals.
    [1, 1, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 1, 1, 0]
])
    # Calculate the sum of each column (one column per orbital)
    column_sums = np.sum(cluster_matrix, axis=0)
    if not np.all(np.isin(column_sums, [0, 1])):
        invalid_columns = np.where((column_sums != 0) & (column_sums != 1))[0]
        raise ValueError(f"Error: Orbitals {invalid_columns} should appear in at most one cluster.")
    if np.all(column_sums == 1):
        raise ValueError(f"Error: The clusters cover all orbitals. Remove one cluster to avoid redundancy.")
    num_clusters = len(cluster_matrix)
else:
    num_clusters = 3 # specify number of clusters

In [18]:
# import new source code

import sys
sys.path.insert(0, '..')
import src.cluster_number_operators as cluster_ops

In [19]:
# ============================================================================
# SECTION 1: Get psi_mps with DMRG using MOs as sites; get 1- and 2-RDMs
# ============================================================================
import pyscf
from chemistry import get_geometry_and_description
from src.dmrg_solver import Block2DMRGSolver, DMRGConfig, solve_or_load_ground_state

# --- Step 1.1: Build molecule and run HF ---
geometry, _ = get_geometry_and_description(molecule, bond_length)
mol = pyscf.M(atom=geometry, basis=basis)
mf = pyscf.scf.RHF(mol)
mf.kernel()

norb, nelec = mol.nao, mol.nelec
h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
g2e = pyscf.ao2mo.full(mol, mf.mo_coeff)
ecore = mol.energy_nuc()

# --- Step 1.2: Run DMRG ---
import tempfile
store_dir = tempfile.mkdtemp(prefix='dmrg_')
solver = Block2DMRGSolver(
    h1e=h1e, g2e=g2e, ecore=ecore,
    n_elec=nelec, spin=0,
    store_dir=store_dir, n_threads=1
)
result = solve_or_load_ground_state(
    solver,
    config=DMRGConfig(max_bond_dim=bond_dim, n_sweeps=n_sweeps),
    reuse=False
)
print(f"DMRG Energy: {result.energy:.10f} Ha")

# --- Step 1.3: Extract RDMs directly from MPS ---
solver._activate()
mps = solver.get_mps()

rdm1 = solver.driver.get_1pdm(mps)
rdm2 = solver.driver.get_2pdm(mps) # .transpose(0, 3, 1, 2) to get chemistry notation

print("Type of rdm1:", type(rdm1))
print("rdm1:\n", rdm1)

converged SCF energy = -74.9633190525401
DMRG Energy: -75.0131547015 Ha
Type of rdm1: <class 'list'>
rdm1:
 [array([[ 9.99998183e-01, -1.94727774e-05,  1.99060119e-11,
         1.84838122e-05, -4.59430726e-10, -3.14225192e-05,
         3.93804985e-10],
       [-1.94727774e-05,  9.96045058e-01,  1.38707459e-11,
        -4.73101064e-03,  3.32105213e-08,  7.05329600e-04,
        -3.23735984e-09],
       [ 1.99060119e-11,  1.38707459e-11,  9.86886751e-01,
        -2.57241875e-10, -7.97831176e-09,  5.46854764e-09,
        -1.92774913e-03],
       [ 1.84838122e-05, -4.73101064e-03, -2.57241875e-10,
         9.91186695e-01,  6.00983371e-08, -1.17536171e-02,
         1.01976409e-08],
       [-4.59430726e-10,  3.32105213e-08, -7.97831176e-09,
         6.00983371e-08,  9.99160680e-01,  9.34731836e-07,
        -9.65566040e-08],
       [-3.14225192e-05,  7.05329600e-04,  5.46854764e-09,
        -1.17536171e-02,  9.34731836e-07,  1.33484602e-02,
         2.37557782e-10],
       [ 3.93804985e-10, -3

In [20]:
# ============================================================================
# SECTION 2: Get best clusters, described via a binary cluster_matrix
# (if custom_cluster_matrix = None). Should use Praveen's beam search.
# ============================================================================


In [21]:
# ============================================================================
# SECTION 3: Get the optimal orbital basis, starting from initial_basis.
# Optimization cost function type is given by type_cost_function.
# ============================================================================

In [22]:
# ============================================================================
# SECTION 4: Get labels of dominant sector and sectors obtained from it by 
# moving up to num_transfers electrons
# ============================================================================

In [23]:
# ============================================================================
# SECTION 5: For small systems, evaluate quality of sector decomposition
# by plotting K_sectors -> ΔE, K_states -> ΔE
# ============================================================================